# 04 - Build Gold Layer

Este notebook constrói a camada Gold a partir da tabela Silver.

## Objetivos

- Criar uma tabela de consumo geral com dados limpos de corridas.
- Criar uma tabela agregada com a média de `total_amount` por mês.
- Criar uma tabela agregada com a média de `passenger_count` por hora no mês de maio.
- Persistir os objetos Gold como tabelas Delta no Unity Catalog.

A camada Gold é orientada ao consumo analítico e responde diretamente às perguntas propostas no case.

---

## 00. Criação da camada Gold

#### 01. Import do Projeto

In [ ]:
import os
import sys
import importlib

PROJECT_ROOT = "/Workspace/Users/lucas12345543@gmail.com/ifood-case"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Files in src:", os.listdir(f"{PROJECT_ROOT}/src"))

for module_name in ["src.build_gold"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.build_gold as gold

print("Silver table:", gold.SILVER_TABLE_NAME)
print("Gold trips table:", gold.GOLD_TRIPS_TABLE_NAME)
print("Gold monthly avg table:", gold.GOLD_MONTHLY_AVG_TABLE_NAME)
print("Gold May hourly passengers table:", gold.GOLD_MAY_HOURLY_PASSENGERS_TABLE_NAME)

from src.build_gold import create_gold

#### 02. Tables Gold

In [ ]:
gold_outputs = create_gold(
    spark=spark,
    mode="overwrite",
)

gold_trips_df = gold_outputs["gold_trips_df"]
monthly_avg_df = gold_outputs["monthly_avg_df"]
may_hourly_passengers_df = gold_outputs["may_hourly_passengers_df"]

print(f"Gold trips records: {gold_trips_df.count()}")

display(gold_trips_df.limit(10))

___

## 01. Validar tabelas Gold

#### 01. Confirmar tabela

In [ ]:
display(
    spark.sql("SHOW TABLES IN workspace.ifood_case")
)

#### 02. Validar tabela Gold de viagens

In [ ]:
gold_trips_table = spark.table("workspace.ifood_case.gold_yellow_taxi_trips")

print(f"Gold trips table records: {gold_trips_table.count()}")

gold_trips_table.printSchema()

display(gold_trips_table.limit(10))

___

## 02. Após essa etapa, a consulta presente em Analysis, já pode ser realizada extraindo a visão de negócio, através do valor do dado gerado pela Pipeline ETL.